# Support Integrity Auditor (SIA) - Colab

**One-time setup before running:**
1. In Google Drive, create a folder `support_integrity_auditor` with two subfolders `src/` and `data/`.
2. Put `config.py`, `pseudo_labeling.py`, `train_pipeline.py` into `src/`.
3. Put `customer_support_tickets.csv` into `data/`.
4. Runtime -> Change runtime type -> **T4 GPU**, then run cells in order.

## Step 0 - Install deps, mount Drive, set paths

In [1]:
!pip -q install "sentence-transformers>=3.0" "datasets>=2.20" "accelerate>=0.33"

In [2]:
import sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT = Path('/content/drive/MyDrive/support_integrity_auditor')
SRC  = PROJECT / 'src'
DATA = PROJECT / 'data'
(PROJECT / 'models').mkdir(parents=True, exist_ok=True)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('Project exists :', PROJECT.exists())
print('src/ files     :', [p.name for p in SRC.glob('*.py')] if SRC.exists() else 'MISSING')
print('data/ files    :', [p.name for p in DATA.glob('*.csv')] if DATA.exists() else 'MISSING')

import torch
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

Mounted at /content/drive
Project exists : True
src/ files     : ['pseudo_labeling.py', 'train_pipeline.py', 'config.py']
data/ files    : ['enhanced_customer_support_data.csv', 'customer_support_tickets.csv']
GPU            : Tesla T4


## Step A - Inspect the data

In [3]:
import pandas as pd
pd.set_option('display.max_colwidth', 80)

df_raw = pd.read_csv(DATA / 'customer_support_tickets.csv')
print('shape:', df_raw.shape)
print('columns:', list(df_raw.columns))
for col in df_raw.columns:
    if 'priority' in col.lower():
        print('priority counts:'); print(df_raw[col].value_counts())
display(df_raw.head(3))

shape: (20000, 12)
columns: ['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject', 'Ticket_Description', 'Issue_Category', 'Priority_Level', 'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours', 'Assigned_Agent', 'Satisfaction_Score']
priority counts:
Priority_Level
Low         7716
Medium      7570
High        3416
Critical    1298
Name: count, dtype: int64


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located? Lay soon message show know m...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time I open the settings tab. Spee...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise plan? Close stand street wear...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5


## Stage 1 - Pseudo-labeling
Fuses `rule + embed` (the two semantic signals). `rtime` is computed and reported but excluded from fusion (it scored at chance on this data). `use_llm=False` since we don't need it; flip to `True` later to add Phi-3 as a third signal for the adversarial-robustness bonus.

In [4]:
from pseudo_labeling import run as stage1_run

INPUT  = DATA / 'customer_support_tickets.csv'
OUTPUT = DATA / 'tickets_pseudolabeled.csv'
REPORT = DATA / 'stage1_report.json'

df, report = stage1_run(
    str(INPUT), str(OUTPUT),
    use_embed=True, use_llm=False, use_rtime=False,
    mismatch_threshold=2, report_path=str(REPORT),
)

Loaded 20000 tickets. Columns: ['ticket_id', 'Customer_Name', 'email', 'subject', 'description', 'ticket_type', 'priority', 'channel', 'Submission_Date', 'resolution_time', 'Assigned_Agent', 'Satisfaction_Score', 'text']
[embed] encoding with sentence-transformers/all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]


=== Stage 1 report ===
{
  "n_tickets": 20000,
  "fusion_weights": {
    "rule": 0.4,
    "embed": 0.6
  },
  "mismatch_threshold": 2,
  "mismatch_distribution": {
    "0": 13524,
    "1": 6476
  },
  "mismatch_type_distribution": {
    "Consistent": 13524,
    "Hidden Crisis": 6439,
    "False Alarm": 37
  },
  "signals_used": [
    "rule",
    "embed"
  ],
  "signals_available": [
    "rule",
    "embed",
    "rtime"
  ],
  "pairwise_agreement": {
    "rule~embed": 0.6698,
    "rule~rtime": 0.2473,
    "embed~rtime": 0.248
  },
  "ablation": {
    "without_rule": {
      "label_agreement_with_full": 0.9815
    },
    "without_embed": {
      "label_agreement_with_full": 0.8794
    }
  }
}

Saved pseudo-labeled data -> /content/drive/MyDrive/support_integrity_auditor/data/tickets_pseudolabeled.csv
Saved report -> /content/drive/MyDrive/support_integrity_auditor/data/stage1_report.json


In [5]:
import json
print('Signals used         :', report['signals_used'])
print('Pairwise agreement   :', report['pairwise_agreement'])
print('Mismatch distribution:', report['mismatch_distribution'])
print('By type              :', report['mismatch_type_distribution'])
print('Ablation             :', json.dumps(report['ablation'], indent=2))

Signals used         : ['rule', 'embed']
Pairwise agreement   : {'rule~embed': 0.6698, 'rule~rtime': 0.2473, 'embed~rtime': 0.248}
Mismatch distribution: {'0': 13524, '1': 6476}
By type              : {'Consistent': 13524, 'Hidden Crisis': 6439, 'False Alarm': 37}
Ablation             : {
  "without_rule": {
    "label_agreement_with_full": 0.9815
  },
  "without_embed": {
    "label_agreement_with_full": 0.8794
  }
}


## Stage 2 - Fine-tune the classifier (DeBERTa-v3-small)
Trains on the T4 (~10-20 min). The metadata (priority, channel, type, resolution hours) is serialized in front of the ticket text, and class imbalance is handled with a weighted loss. The model is saved to your Drive so it survives a runtime reset.

**Verification thresholds:** accuracy >= 0.83, macro-F1 >= 0.82, per-class recall >= 0.78.

In [6]:
from train_pipeline import run as stage2_run

MODEL_DIR = str(PROJECT / 'models' / 'sia_deberta')
METRICS   = str(DATA / 'stage2_metrics.json')

metrics = stage2_run(
    input_csv=str(OUTPUT),     # tickets_pseudolabeled.csv from Stage 1
    model_dir=MODEL_DIR,
    metrics_path=METRICS,
    epochs=3, batch_size=16, max_len=256,
)

Dataset: 20000 rows | label balance:
  {0: 13524, 1: 6476}
Split: train=14000 val=3000 test=3000


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== Stage 2 held-out test metrics ===
{
  "accuracy": 0.979,
  "macro_f1": 0.9758798094734664,
  "recall_consistent": 0.9901380670611439,
  "recall_mismatch": 0.9557613168724279,
  "confusion_matrix": [
    [
      2008,
      20
    ],
    [
      43,
      929
    ]
  ],
  "thresholds": {
    "accuracy_ge_0.83": true,
    "macro_f1_ge_0.82": true,
    "per_class_recall_ge_0.78": true
  },
  "verified": true
}
VERIFIED


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved model -> /content/drive/MyDrive/support_integrity_auditor/models/sia_deberta
Saved metrics -> /content/drive/MyDrive/support_integrity_auditor/data/stage2_metrics.json


In [7]:
import json
print('Accuracy          :', round(metrics['accuracy'], 4))
print('Macro F1          :', round(metrics['macro_f1'], 4))
print('Recall (Consistent):', round(metrics['recall_consistent'], 4))
print('Recall (Mismatch) :', round(metrics['recall_mismatch'], 4))
print('Confusion matrix  :', metrics['confusion_matrix'])
print('Thresholds        :', json.dumps(metrics['thresholds'], indent=2))
print('VERIFIED          :', metrics['verified'])

Accuracy          : 0.979
Macro F1          : 0.9759
Recall (Consistent): 0.9901
Recall (Mismatch) : 0.9558
Confusion matrix  : [[2008, 20], [43, 929]]
Thresholds        : {
  "accuracy_ge_0.83": true,
  "macro_f1_ge_0.82": true,
  "per_class_recall_ge_0.78": true
}
VERIFIED          : True


In [8]:
%%writefile /content/drive/MyDrive/support_integrity_auditor/src/dossier.py
"""Stage 3 - deterministic, hallucination-free Evidence Dossier builder."""
from __future__ import annotations
import json
import pandas as pd
from config import ORD_TO_PRIORITY


def _keyword_evidence(rule_evidence):
    items = []
    try:
        ev = (json.loads(rule_evidence)
              if isinstance(rule_evidence, str) else (rule_evidence or {}))
    except Exception:
        ev = {}
    for tier, terms in ev.items():
        for term in terms:
            items.append({"signal": "keyword", "value": term, "weight": tier})
    return items


def _safe_int(v):
    try:
        if pd.isna(v):
            return None
        return int(v)
    except Exception:
        return None


def build_dossier(row, confidence=None):
    inferred_ord = _safe_int(row.get("inferred_ord"))
    inferred = ORD_TO_PRIORITY.get(inferred_ord, str(inferred_ord))
    assigned = str(row.get("priority"))
    delta = _safe_int(row.get("severity_delta"))
    mtype = row.get("mismatch_type")

    evidence = []
    evidence += _keyword_evidence(row.get("rule_evidence"))
    sev_embed = row.get("sev_embed")
    if sev_embed is not None and not pd.isna(sev_embed):
        closest = ORD_TO_PRIORITY.get(round(float(sev_embed)), "")
        evidence.append({"signal": "embedding",
                         "value": round(float(sev_embed), 2),
                         "interpretation": f"text semantically closest to {closest}-severity tickets"})
    rt = row.get("resolution_time")
    if rt is not None and not pd.isna(rt):
        evidence.append({"signal": "resolution_time",
                         "value": rt,
                         "interpretation": "reported resolution hours (context only; not weighted)"})

    kws = [e["value"] for e in evidence if e["signal"] == "keyword"][:4]
    kw_str = ", ".join(f'"{k}"' for k in kws)
    embed_label = (ORD_TO_PRIORITY.get(round(float(sev_embed)))
                   if sev_embed is not None and not pd.isna(sev_embed) else None)
    direction = ("under-prioritized" if mtype == "Hidden Crisis"
                 else "over-prioritized" if mtype == "False Alarm" else "consistent")

    if delta is not None:
        s1 = (f"Assigned priority is {assigned}, but the fused signals infer "
              f"{inferred} severity (delta {delta:+d}). ")
    else:
        s1 = f"Assigned priority is {assigned}; the fused signals infer {inferred} severity. "

    bits = []
    if kws:
        bits.append(f"escalation cues {kw_str}")
    if embed_label:
        bits.append(f"an embedding profile nearest {embed_label}-severity content")
    s2 = ("The text shows " + " and ".join(bits) + ". ") if bits else ""

    if mtype in ("Hidden Crisis", "False Alarm"):
        rel = "exceeds" if mtype == "Hidden Crisis" else "falls below"
        s3 = (f"Because the inferred severity {rel} the assigned {assigned} label, "
              f"this is flagged as a {mtype} ({direction}).")
    else:
        s3 = "Inferred and assigned severities are aligned."
    analysis = (s1 + s2 + s3).strip()

    return {
        "ticket_id": row.get("ticket_id"),
        "assigned_priority": assigned,
        "inferred_severity": inferred,
        "mismatch_type": mtype,
        "severity_delta": delta,
        "feature_evidence": evidence,
        "constraint_analysis": analysis,
        "confidence": round(float(confidence), 4) if confidence is not None else None,
    }

Writing /content/drive/MyDrive/support_integrity_auditor/src/dossier.py


In [9]:
import torch, json, importlib
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from train_pipeline import build_input
import dossier; importlib.reload(dossier)
from dossier import build_dossier

device = 'cuda' if torch.cuda.is_available() else 'cpu'
clf_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
clf = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()

def predict_df(frame, batch_size=64, max_len=256):
    texts = frame.apply(build_input, axis=1).tolist()
    preds, confs = [], []
    for i in range(0, len(texts), batch_size):
        enc = clf_tok(texts[i:i+batch_size], truncation=True, max_length=max_len,
                      padding=True, return_tensors='pt').to(device)
        with torch.no_grad():
            prob = torch.softmax(clf(**enc).logits, dim=-1)
        preds.extend(prob.argmax(-1).cpu().tolist())
        confs.extend(prob.max(-1).values.cpu().tolist())
    return np.array(preds), np.array(confs)

full = pd.read_csv(OUTPUT)              # tickets_pseudolabeled.csv (has evidence cols)
pred, conf = predict_df(full)
full['pred_mismatch'], full['pred_confidence'] = pred, conf

flagged = full[full['pred_mismatch'] == 1].copy()
dossiers = [build_dossier(r, c) for (_, r), c in zip(flagged.iterrows(), flagged['pred_confidence'])]

with open(DATA / 'dossiers.json', 'w') as f:
    json.dump(dossiers, f, indent=2)
print(f"Classified {len(full)} tickets; {len(flagged)} flagged as mismatch.")
print(f"Saved {len(dossiers)} dossiers -> {DATA / 'dossiers.json'}")

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Classified 20000 tickets; 6314 flagged as mismatch.
Saved 6314 dossiers -> /content/drive/MyDrive/support_integrity_auditor/data/dossiers.json


In [10]:
import json
shown = {}
for d in dossiers:
    if d['mismatch_type'] in ('Hidden Crisis', 'False Alarm') and d['mismatch_type'] not in shown:
        shown[d['mismatch_type']] = d
    if len(shown) == 2:
        break
for t, d in shown.items():
    print(f"\n===== {t} =====")
    print(json.dumps(d, indent=2))


===== Hidden Crisis =====
{
  "ticket_id": "TKT-100003",
  "assigned_priority": "Low",
  "inferred_severity": "High",
  "mismatch_type": "Hidden Crisis",
  "severity_delta": 2,
  "feature_evidence": [
    {
      "signal": "keyword",
      "value": "failed",
      "weight": "high"
    },
    {
      "signal": "embedding",
      "value": 2.51,
      "interpretation": "text semantically closest to Critical-severity tickets"
    },
    {
      "signal": "resolution_time",
      "value": 41,
      "interpretation": "reported resolution hours (context only; not weighted)"
    }
  ],
  "constraint_analysis": "Assigned priority is Low, but the fused signals infer High severity (delta +2). The text shows escalation cues \"failed\" and an embedding profile nearest Critical-severity content. Because the inferred severity exceeds the assigned Low label, this is flagged as a Hidden Crisis (under-prioritized).",
  "confidence": 0.9999
}


In [11]:
%%writefile /content/drive/MyDrive/support_integrity_auditor/src/predict.py
"""SIA inference - raw tickets CSV -> predictions + Evidence Dossiers."""
from __future__ import annotations
import argparse
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pseudo_labeling
from dossier import build_dossier

META_COLS = ["priority", "channel", "ticket_type", "resolution_time"]


def build_input(row) -> str:
    parts = []
    for c in META_COLS:
        v = row.get(c)
        if v is not None and not (isinstance(v, float) and pd.isna(v)):
            parts.append(f"{c}: {v}")
    text = "" if pd.isna(row.get("text")) else str(row.get("text"))
    return (" | ".join(parts) + " | " + text).strip(" |")


def load_model(model_dir):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    return tok, model, device


def classify(frame, tok, model, device, batch_size=64, max_len=256):
    texts = frame.apply(build_input, axis=1).tolist()
    preds, confs = [], []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            prob = torch.softmax(model(**enc).logits, dim=-1)
        preds.extend(prob.argmax(-1).cpu().tolist())
        confs.extend(prob.max(-1).values.cpu().tolist())
    return np.array(preds), np.array(confs)


def run(input_csv, model_dir, predictions_out="predictions.csv",
        dossiers_out="dossiers.json", use_embed=True, use_llm=False):
    labeled, _ = pseudo_labeling.run(
        input_csv, "/tmp/_sia_labeled.csv", use_embed=use_embed, use_llm=use_llm)
    tok, model, device = load_model(model_dir)
    pred, conf = classify(labeled, tok, model, device)
    labeled["pred_mismatch"] = pred
    labeled["pred_confidence"] = conf
    labeled["pred_label"] = np.where(pred == 1, "Mismatch", "Consistent")

    cols = ["ticket_id", "priority", "inferred_ord", "severity_delta",
            "mismatch_type", "pred_label", "pred_mismatch", "pred_confidence"]
    cols = [c for c in cols if c in labeled.columns]
    labeled[cols].to_csv(predictions_out, index=False)

    flagged = labeled[labeled["pred_mismatch"] == 1]
    dossiers = [build_dossier(r, c)
                for (_, r), c in zip(flagged.iterrows(), flagged["pred_confidence"])]
    with open(dossiers_out, "w") as f:
        json.dump(dossiers, f, indent=2)

    print(f"\nPredictions -> {predictions_out}  ({len(labeled)} tickets)")
    print(f"Dossiers    -> {dossiers_out}  ({len(dossiers)} flagged as mismatch)")
    return labeled, dossiers


def main(argv=None):
    ap = argparse.ArgumentParser(description="SIA inference: CSV -> predictions + dossiers")
    ap.add_argument("--input", required=True)
    ap.add_argument("--model_dir", required=True)
    ap.add_argument("--predictions", default="predictions.csv")
    ap.add_argument("--dossiers", default="dossiers.json")
    ap.add_argument("--no_embed", action="store_true")
    ap.add_argument("--use_llm", action="store_true")
    args = ap.parse_args(argv)
    run(args.input, args.model_dir, args.predictions, args.dossiers,
        use_embed=not args.no_embed, use_llm=args.use_llm)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/support_integrity_auditor/src/predict.py


In [12]:
!python /content/drive/MyDrive/support_integrity_auditor/src/predict.py \
  --input /content/drive/MyDrive/support_integrity_auditor/data/customer_support_tickets.csv \
  --model_dir /content/drive/MyDrive/support_integrity_auditor/models/sia_deberta \
  --predictions /content/drive/MyDrive/support_integrity_auditor/data/predictions.csv \
  --dossiers /content/drive/MyDrive/support_integrity_auditor/data/dossiers.json

Loaded 20000 tickets. Columns: ['ticket_id', 'Customer_Name', 'email', 'subject', 'description', 'ticket_type', 'priority', 'channel', 'Submission_Date', 'resolution_time', 'Assigned_Agent', 'Satisfaction_Score', 'text']
[embed] encoding with sentence-transformers/all-MiniLM-L6-v2 ...
Loading weights: 100% 103/103 [00:00<00:00, 743.21it/s]
Batches: 100% 313/313 [00:06<00:00, 45.25it/s]

=== Stage 1 report ===
{
  "n_tickets": 20000,
  "fusion_weights": {
    "rule": 0.4,
    "embed": 0.6
  },
  "mismatch_threshold": 2,
  "mismatch_distribution": {
    "0": 13524,
    "1": 6476
  },
  "mismatch_type_distribution": {
    "Consistent": 13524,
    "Hidden Crisis": 6439,
    "False Alarm": 37
  },
  "signals_used": [
    "rule",
    "embed"
  ],
  "signals_available": [
    "rule",
    "embed",
    "rtime"
  ],
  "pairwise_agreement": {
    "rule~embed": 0.6698,
    "rule~rtime": 0.2473,
    "embed~rtime": 0.248
  },
  "ablation": {
    "without_rule": {
      "label_agreement_with_full": 0

In [13]:
from huggingface_hub import login, create_repo, upload_folder, whoami
from getpass import getpass

TOKEN = getpass("Paste your HF WRITE token: ")
login(token=TOKEN, add_to_git_credential=False)

username = whoami(token=TOKEN)["name"]      # fails loudly if the token is invalid
print("Authenticated as:", username)

REPO_ID = f"{username}/sia-deberta"
create_repo(REPO_ID, exist_ok=True, token=TOKEN)
upload_folder(
    folder_path=MODEL_DIR, repo_id=REPO_ID, token=TOKEN,
    ignore_patterns=["checkpoint-*", "checkpoint-*/*", "runs/*", "optimizer.pt"],
)
print("Pushed to:", REPO_ID)

Paste your HF WRITE token: ··········
Authenticated as: Darkrai17


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...deberta/model.safetensors:  13%|#2        | 72.0MB /  568MB            

  ...deberta/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to: Darkrai17/sia-deberta
